# 04 — Spinal cord analysis with Spinal Cord Toolbox (SCT)

Spinal cord involvement is common in MS and often clinically more relevant than brain lesions for disability. Yet cord imaging is underused because pipelines are tricky.

**Spinal Cord Toolbox (SCT)** is the de-facto open-source pipeline. We'll:
1. Segment the cord
2. Label vertebrae
3. Compute cross-sectional area (CSA) per vertebral level — a standard atrophy metric in MS

In [ ]:
from pathlib import Path
DATA = Path('/neurodesktop-storage/ms-workshop-data/sub-001/cord')
OUT  = Path('/neurodesktop-storage/ms-workshop-out/sub-001/cord')
OUT.mkdir(parents=True, exist_ok=True)
T2 = DATA / 'sub-001_T2w_cord.nii.gz'
print(T2.exists())

## 1. Cord segmentation

In [ ]:
%%bash -s "$T2" "$OUT"
ml load spinalcordtoolbox
T2=$1; OUT=$2
sct_deepseg_sc -i $T2 -c t2 -ofolder $OUT

## 2. Vertebrae labelling

In [ ]:
%%bash -s "$T2" "$OUT"
ml load spinalcordtoolbox
T2=$1; OUT=$2
SEG=$OUT/$(basename ${T2%.nii.gz})_seg.nii.gz
sct_label_vertebrae -i $T2 -s $SEG -c t2 -ofolder $OUT

## 3. CSA per vertebral level

In [ ]:
%%bash -s "$T2" "$OUT"
ml load spinalcordtoolbox
T2=$1; OUT=$2
SEG=$OUT/$(basename ${T2%.nii.gz})_seg.nii.gz
LABEL=$OUT/$(basename ${T2%.nii.gz})_seg_labeled.nii.gz
sct_process_segmentation -i $SEG -vert 2:5 -vertfile $LABEL \
  -o $OUT/CSA.csv -perlevel 1

In [ ]:
import pandas as pd
df = pd.read_csv(OUT/'CSA.csv')
df

## Clinical relevance

- **Upper cervical CSA (C2–C3)** is the most replicated cord atrophy biomarker in MS.
- Cord atrophy correlates with EDSS disability *better* than brain atrophy in many cohorts.
- For longitudinal monitoring, use the same scanner/protocol/pipeline.

Reference: De Leener et al., *NeuroImage* 2017 — SCT main paper.